# PR Risk Classifier — What the Numbers Actually Mean

A walkthrough of `evaluation_report.md`, with the formula behind every number and this run's actual figures plugged in. 273 pull requests were labeled by a human (the "true" class) and also scored by the model (the "predicted" class). The three classes, from most to least dangerous, are:

- **CT1** — high risk, needs a real reviewer
- **CT2** — medium risk
- **CT3** — low risk, safe to bulk-approve

## 1. Confusion matrix — who got sorted where

Think of it as three baskets (CT1/CT2/CT3) and we're checking, for every PR that truly belongs in basket *CT1*, which basket the model actually dropped it into.

| | pred CT1 | pred CT2 | pred CT3 |
| --- | --- | --- | --- |
| true CT1 | 122 | 2 | 0 |
| true CT2 | 17 | 114 | 0 |
| true CT3 | 18 | 0 | 0 |

- Of 124 truly high-risk PRs, the model correctly flagged 122 and only missed 2 (into CT2).
- Of 131 truly medium-risk PRs, it correctly caught 114, but 17 got bumped up to CT1 (over-cautious, not dangerous).
- Of 18 truly low-risk PRs, **none** were recognized as low-risk — all 18 got escalated to CT1. The model currently never predicts CT3 at all.

That last point matters a lot for everything below: the model is safe (it never lets something dangerous slip through unnoticed) but it's currently "trigger-happy" — pushing everything uncertain up to CT1 rather than confidently calling something safe.

## 2. Precision, Recall, F1 — how good is each basket

**Precision** — of everything the model *called* class CT1, what fraction really was CT1? (Are we crying wolf?)

$$
\text{Precision} = \frac{\text{Correctly called CT1}}{\text{Everything called CT1}}
$$

**Recall** — of everything that *truly is* class CT1, what fraction did the model catch? (Do we miss anything?)

$$
\text{Recall} = \frac{\text{Correctly called CT1}}{\text{Everything that is really CT1}}
$$

**F1** — one number that balances the two (only high when *both* precision and recall are decent):

$$
F_1 = \frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

**Worked example for CT1**, straight from the confusion matrix above:
- Called CT1 total: 122 + 17 + 18 = 157. Correctly: 122. → Precision = 122 / 157 = **0.78**
- Truly CT1 total: 124. Correctly caught: 122. → Recall = 122 / 124 = **0.98**
- F1 = 2 × 0.78 × 0.98 / (0.78 + 0.98) = **0.87**

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| CT1 | 0.78 | 0.98 | 0.87 | 124 |
| CT2 | 0.98 | 0.87 | 0.92 | 131 |
| CT3 | 0.00 | 0.00 | 0.00 | 18 |

**Macro F1** just averages the three F1 scores equally, so the small CT3 group can't get lost in the crowd:

$$
\text{Macro } F_1 = \frac{F_1(\text{CT1}) + F_1(\text{CT2}) + F_1(\text{CT3})}{3} = \frac{0.87 + 0.92 + 0.00}{3} = \mathbf{0.597}
$$

**The headline number to remember:** Recall(CT1) = **98.4%** — out of every 1,000 genuinely dangerous PRs, this model catches about 984 of them without a human pointing it out first.

## 3. ROC-AUC — can the model tell risky from safe, even before picking a cutoff

The model doesn't just say "CT1" — it outputs a confidence score (like "73% sure this is CT1"). ROC-AUC asks: if you grabbed one truly-CT1 PR and one truly-not-CT1 PR at random, how often does the model's score rank the CT1 one higher?

$$
\text{AUC} = P\big(\text{score of a random true-CT1 example} > \text{score of a random non-CT1 example}\big)
$$

- AUC = 1.00 → perfect separation, always ranks correctly
- AUC = 0.50 → a coin flip, no better than guessing

| Class | AUC |
| --- | --- |
| CT1 | 0.991 |
| CT2 | 0.991 |
| CT3 | 1.000 |

**Macro AUC** = average of the three = **0.994**.

**Why CT3 gets a perfect 1.000 AUC despite 0% recall/precision above:** the model's *raw confidence score* for CT3 is actually excellent at telling CT3 apart from the rest — it "knows" internally which PRs are low-risk. The problem is the decision rule sitting on top of that score is currently too conservative and reroutes anything CT3-flavored up to CT1 anyway, as a safety net. That's a much cheaper fix (retune a threshold) than retraining the model.

## 4. Calibration — when it says 70% sure, is it actually right 70% of the time?

This isn't a single number but a picture (`calibration_curve.png`). For each class, we bucket predictions by their confidence score (e.g., "all PRs the model was 60–70% sure were CT1") and check how often those PRs really were CT1:

$$
\text{Observed frequency in a bucket} = \frac{\text{\# actually class CT1 in that confidence bucket}}{\text{\# total PRs in that confidence bucket}}
$$

Plot that against the average confidence in the bucket. If dots sit on the diagonal line, the confidence score can be trusted at face value ("90% confident" really does mean right 9 times out of 10). Dots above the line = underconfident; below the line = overconfident (dangerous, since it means the model is more unsure than it admits).

## Summary

The model almost never lets a truly dangerous PR through unnoticed (98.4% recall on CT1, 0% bulk-approved-when-risky). The price it pays for that safety is being over-cautious — it currently treats every "safe" PR the same as a "medium" one and routes them all up for review instead of confidently bulk-approving them, which is why CT3 precision/recall sit at 0% despite the model's underlying signal for CT3 being excellent (AUC = 1.00). That's a tuning problem, not a fundamental modeling problem.